In [ ]:
# --- parameters (patch_notebook_params.py) ---
MAX_EPOCHS = 2000
N_SAMPLES = 10           # run_metrics.py forces 50 for generative methods
RESET_TRAINING = False
CUDA_VISIBLE_DEVICES = "0"
METRICS_CSV = "results/metrics.csv"
SKIP_TRAINING = False    # run_metrics.py sets this True: load weights from
                         # the checkpoint directly instead of calling
                         # trainer.fit(), which can silently retrain for the
                         # full schedule if the checkpoint does not cleanly
                         # resume to exactly MAX_EPOCHS.


In [ ]:
# --- epoch heartbeat (patch_notebook_params.py) ---
import pytorch_lightning as _pl

class EpochHeartbeat(_pl.Callback):
    """Prints one clear progress line every `every_n_epochs` epochs, so
    sbatch logs show training progress without the noise of a per-batch
    tqdm progress bar (which doesn't render well once redirected to a
    plain log file).    """

    def __init__(self, every_n_epochs: int = 1):
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        if epoch % self.every_n_epochs != 0 and epoch != trainer.max_epochs:
            return
        parts = []
        for k, v in sorted(trainer.callback_metrics.items()):
            try:
                parts.append(f'{k}={float(v):.4f}')
            except (TypeError, ValueError):
                pass
        print(f'[progress] epoch {epoch}/{trainer.max_epochs} | ' + ' | '.join(parts), flush=True)


## Consistency Model — Sea Ice Concentration (SIC)

In [ ]:
!nvidia-smi

In [ ]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

In [ ]:
import json
import math
import os
import zipfile
import glob
import importlib
from dataclasses import asdict, dataclass
from typing import Any, Callable, List, Optional, Tuple, Union

import numpy as np
import xarray as xr
import pandas as pd
import torch
from einops import rearrange
from einops.layers.torch import Rearrange
import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer, seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from matplotlib import pyplot as plt
from torch import Tensor, nn
from torch.nn import functional as F
from torchinfo import summary

import sys
sys.path.append('../..')     # -> consistency/  (consistency_models, spectral_utils)
sys.path.append('../../..')  # -> 4dvarnet-starter-devs/

import consistency_models.utils
import consistency_models.consistency_models_CM
importlib.reload(consistency_models.utils)
importlib.reload(consistency_models.consistency_models_CM)

from consistency_models.consistency_models_CM import (
    ConsistencySamplingAndEditingFewSteps_TimeEmbedding,
    ConsistencyTrainingFewSteps_TimeEmbedding,
    ema_decay_rate_schedule,
)
from consistency_models.utils import update_ema_model_, pseudo_huber_loss

try:
    from properscoring import crps_ensemble
    HAS_PROPERSCORING = True
except ImportError:
    HAS_PROPERSCORING = False
    print("properscoring not installed -- CRPS disabled.")

# --- method-specific imports (added by generate_missing_notebooks.py) ---
from consistency_models.consistency_models_DynCM import (
    ConsistencyTrainingDynamicalSystems,
    ConsistencySamplingAndEditingDynamicalSystems,
    model_dynamical_systems_forward_wrapper,
    compute_sigma_spinup,
)
from consistency_models.consistency_models_CM import (
    ema_decay_rate_schedule,
)


## ⚠️ ACTION REQUIRED — automatically generated notebook

This notebook (`DynCM` / `SIC`) was assembled by `sbatch_submission/generate_missing_notebooks.py`, combining the data loading of xp `SIC` with the method code of the GP reference notebook. Points to validate before a production run:

- **LOG_DIR** automatically set to `logs_CT_DynCM_sic` — check it doesn't collide with an existing run.
- **sigma_max / SIGMA_NOISE**: value carried over as-is from the GP source notebook — needs to be re-tuned to this variable's physical scale (cf. SIC ≈ 1.0, SSH_GF ≈ 10, GP to verify) before any production run.
- **cond_channels**: `UNetConfig(channels=C)` was automatically rewritten to `UNetConfig(channels=C, cond_channels=2*C)` (factor inferred from the existing SIC/CM notebook). Verify this factor is correct for the config class used by **DynCM** (may differ from CM's) and that the forward pass actually concatenates the right conditioning fields.
- **Masked loss (NaN)**: this xp has NaN gaps in the target. The matching FM notebook (`Notebook_flowmatching_model_FM_sic.ipynb`, class `LitFlowMatchingSIC.training_step`) already implements masking via `masked_average` — **this generated notebook did NOT reproduce it automatically** for this method's Lightning class. Must be done before any real training run, otherwise the model will train on NaN noise.

## DataModule -- SIC (ASIP / OSISAF)

In [ ]:
sys.path.insert(0, '../../..')
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import functools as ft
from src.dataloader_SIC import BaseDataModule, TrainingItem_4da

T_CROP = 9

class CroppedDataModule(BaseDataModule):
    def __init__(self, *args, t_crop=9, **kwargs):
        super().__init__(*args, **kwargs)
        self.t_crop = t_crop

    def build_batch(self):
        # ASIP/input/osisaf are already in [0, 1] in the NetCDF file — no rescaling needed
        return ft.partial(ft.reduce, lambda i, f: f(i), [
            TrainingItem_4da._make,
        ])

    def setup(self, stage='test'):
        super().setup(stage)
        for ds in [self.train_ds, self.val_ds, self.test_ds]:
            ds.db = ds.db.isel(time=slice(-self.t_crop, None))

datamodule = CroppedDataModule(
    asip_paths="../../../data/asip_database_daw15_sparse.nc",
    split_train=slice(0, 300),
    split_val=slice(300, 350),
    split_test=slice(350, 393),
    da=True,
    norm_stats=[0, 1],
    norm_stats_covs=[
        {'t2m': 270.08, 'istl1': 267.68, 'siconc': 0, 'sst': 276.97, 'skt': 270.50},
        {'t2m': 14.67,  'istl1': 7.80,  'siconc': 1, 'sst': 6.82,   'skt': 15.21},
    ],
    t_crop=T_CROP,
)
datamodule.setup()

sample = datamodule.train_ds[0]
C = sample.asip.shape[0]
datamodule.window_size = C  # shim: window_size only exists on the GP/SPDE
# reference datamodule this notebook's later cells were copied from -- the
# ASIP CroppedDataModule used here doesn't define it, so later cells that
# read datamodule.window_size would otherwise crash with AttributeError.
H, W = sample.asip.shape[1], sample.asip.shape[2]
print(f"C={C}, H={H}, W={W}")
print(f"Train: {len(datamodule.train_ds)}, Val: {len(datamodule.val_ds)}, Test: {len(datamodule.test_ds)}")
print(f"asip: {sample.asip.shape}  input: {sample.input.shape}  osisaf: {sample.osisaf.shape}")
print(f"asip range: [{np.nanmin(sample.asip):.3f}, {np.nanmax(sample.asip):.3f}]")
print(f"asip std: {np.nanstd(sample.asip):.4f}")
print(f"NaN fraction in asip: {np.isnan(sample.asip).mean()*100:.1f}%")

fig, axes = plt.subplots(3, min(C, 5), figsize=(3 * min(C, 5), 9))
for t_idx in range(min(C, 5)):
    axes[0, t_idx].imshow(sample.asip[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[0, t_idx].set_title(f't={t_idx}', fontsize=8); axes[0, t_idx].axis('off')
    axes[1, t_idx].imshow(sample.input[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[1, t_idx].axis('off')
    axes[2, t_idx].imshow(sample.osisaf[t_idx], vmin=0, vmax=1, cmap='Blues_r')
    axes[2, t_idx].axis('off')
axes[0, 0].set_ylabel('ASIP (target)', fontsize=10)
axes[1, 0].set_ylabel('Input (gappy)', fontsize=10)
axes[2, 0].set_ylabel('OSISAF (cond.)', fontsize=10)
plt.tight_layout()
plt.show()

### UNet Building Blocks

The UNet takes:
- **x**: `(B, 1, H, W)` — current state at diffusion time $t$ (single physical field)
- **y**: `(B, C, H, W)` — full observation window (all $N$ time steps) — `NaN` = unobserved
- **mask**: `(B, C, H, W)` — binary mask derived from `y`
- **t, t'**: `(B,)` — current and target diffusion times

Total input channels = $1 + C + C = 2C + 1$.  
Output = `(B, 1, H, W)` — predicted state at $t'$.

Spatial size 100×100 is padded to the nearest multiple of 8 before the forward pass.

In [ ]:
def GroupNorm(channels: int) -> nn.GroupNorm:
    return nn.GroupNorm(num_groups=min(32, channels // 4), num_channels=channels)


class SelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.dropout = dropout
        self.qkv_projection = nn.Sequential(
            GroupNorm(in_channels),
            nn.Conv2d(in_channels, 3 * in_channels, kernel_size=1, bias=False),
            Rearrange("b (i h d) x y -> i b h (x y) d", i=3, h=n_heads),
        )
        self.output_projection = nn.Sequential(
            Rearrange("b h l d -> b l (h d)"),
            nn.Linear(in_channels, out_channels, bias=False),
            Rearrange("b l d -> b d l"),
            GroupNorm(out_channels),
            nn.Dropout1d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        q, k, v = self.qkv_projection(x).unbind(dim=0)
        output = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False
        )
        output = self.output_projection(output)
        output = rearrange(output, "b c (x y) -> b c x y", x=x.shape[-2], y=x.shape[-1])
        return output + self.residual_projection(x)


class UNetBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 time_level_channels: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.input_projection = nn.Sequential(
            GroupNorm(in_channels), nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.time_level_projection = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(time_level_channels, out_channels, kernel_size=1),
        )
        self.output_projection = nn.Sequential(
            GroupNorm(out_channels), nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor, time_level: Tensor) -> Tensor:
        h = self.input_projection(x)
        h = h + self.time_level_projection(time_level)
        return self.output_projection(h) + self.residual_projection(x)


class UNetBlockWithSelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 time_level_channels: int, n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.unet_block = UNetBlock(in_channels, out_channels, time_level_channels, dropout)
        self.self_attention = SelfAttention(out_channels, out_channels, n_heads, dropout)

    def forward(self, x: Tensor, time_level: Tensor) -> Tensor:
        return self.self_attention(self.unet_block(x, time_level))


class Downsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            Rearrange("b c (h ph) (w pw) -> b (c ph pw) h w", ph=2, pw=2),
            nn.Conv2d(4 * channels, channels, kernel_size=1),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


class Upsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            nn.Upsample(scale_factor=2.0, mode="nearest"),
            nn.Conv2d(channels, channels, kernel_size=3, padding="same"),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


class TimeEmbedding(nn.Module):
    """Fourier embedding for a scalar t ∈ [0,1]."""
    def __init__(self, channels: int, scale: float = 16.0) -> None:
        super().__init__()
        self.W = nn.Parameter(torch.randn(channels // 2) * scale, requires_grad=False)
        self.projection = nn.Sequential(
            nn.Linear(channels, 4 * channels),
            nn.SiLU(),
            nn.Linear(4 * channels, channels),
            Rearrange("b c -> b c () ()"),
        )

    def forward(self, x: Tensor) -> Tensor:
        h = x[:, None] * self.W[None, :] * 2 * torch.pi
        h = torch.cat([torch.sin(h), torch.cos(h)], dim=-1)
        return self.projection(h)


# ---- Spatial padding (100×100 → 104×104, multiple of 8) ----
def _pad_to_multiple(x: Tensor, multiple: int = 8) -> Tuple[Tensor, Tuple[int, int, int, int]]:
    _, _, H, W = x.shape
    pad_h = (multiple - H % multiple) % multiple
    pad_w = (multiple - W % multiple) % multiple
    padding = (0, pad_w, 0, pad_h)
    return F.pad(x, padding, mode="reflect"), padding


def _unpad(x: Tensor, padding: Tuple[int, int, int, int]) -> Tensor:
    _, pad_w, _, pad_h = padding
    H, W = x.shape[-2], x.shape[-1]
    return x[..., :H - pad_h if pad_h else H, :W - pad_w if pad_w else W]

### UNet

Input: `cat(x, y_filled, mask_y, w_obs)` → $3C+1$ channels, output: `1` channel.  
Two separate `TimeEmbedding` for $t$ and $t'$, concatenated → $2 \times$ `time_level_channels` conditioning.

**Key design** — observation weight vector $\mathbf{w}(t) \in [0,1]^C$:  
The UNet computes this vector internally from $t$, using exactly the same interpolation rule as the GT target in `_make_regime_input`:
- **Spin-up** ($t \le t_\text{IC}$): $w_0 = 1$, others 0 → only the IC obs is "local"  
- **Physical** ($t \in [t_k, t_{k+1}]$): $w_k = 1-\lambda$, $w_{k+1} = \lambda$, others 0

This weight vector is broadcast spatially and appended as $C$ additional input channels. The model thereby always knows which obs channels are temporally relevant to the current step — **no channel permutation needed**, and the correspondence is consistent with the supervised training targets.


In [ ]:
@dataclass
class UNetConfig:
    # channels = C = window_size (number of physical time steps)
    channels: int = 5
    time_level_channels: int = 128
    time_level_scale: float = 16.0
    n_heads: int = 8
    top_blocks_channels: Tuple[int, ...] = (64, 64)
    top_blocks_n_blocks_per_resolution: Tuple[int, ...] = (2, 2)
    top_blocks_has_resampling: Tuple[bool, ...] = (True, True)
    top_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)
    mid_blocks_channels: Tuple[int, ...] = (128, 256)
    mid_blocks_n_blocks_per_resolution: Tuple[int, ...] = (4, 4)
    mid_blocks_has_resampling: Tuple[bool, ...] = (True, False)
    mid_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)


def _compute_obs_weights(t: Tensor, C: int) -> Tensor:
    """
    Compute the C-dimensional observation weight vector w(t) for each batch element.

    Uses the same temporal interpolation rule as _make_regime_input:
      - spin-up  (t ≤ t_IC = 1/C) : w = [1, 0, ..., 0]   (only IC frame relevant)
      - physical (t > t_IC)        : w_k = 1-λ, w_{k+1} = λ, others = 0
        where k = lower physical frame index and λ = fractional position in [t_k, t_{k+1}]

    Parameters
    ----------
    t : (B,) — current diffusion times, in [0, 1]
    C : int   — number of physical frames (= window_size)

    Returns
    -------
    w : (B, C) float tensor, values in [0, 1], sums to 1 per batch element
    """
    B      = t.shape[0]
    t_IC   = 1.0 / C
    device, dtype = t.device, t.dtype

    physical_steps = torch.linspace(t_IC, 1.0, C, device=device, dtype=dtype)  # (C,)

    # For each b, find the lower-bound physical frame index (vectorised)
    t_clamped = t.clamp(t_IC, 1.0).unsqueeze(1)                         # (B, 1)
    idx = torch.searchsorted(
        physical_steps.unsqueeze(0).expand(B, -1).contiguous(),
        t_clamped,
    ).squeeze(1).clamp(1, C - 1)                                         # (B,)  in [1, C-1]

    b_idx = torch.arange(B, device=device)
    dt    = (physical_steps[idx] - physical_steps[idx - 1]).clamp(min=1e-8)
    lam   = ((t.clamp(t_IC, 1.0) - physical_steps[idx - 1]) / dt).clamp(0., 1.)  # (B,)

    w = torch.zeros(B, C, device=device, dtype=dtype)
    w[b_idx, idx - 1] = 1.0 - lam
    w[b_idx, idx]     = lam

    # Override spin-up elements: only IC frame (index 0) is relevant
    spinup = (t <= t_IC)          # (B,)  bool
    w[spinup]    = 0.0
    w[spinup, 0] = 1.0

    return w  # (B, C)


class UNet(nn.Module):
    """
    UNet for the unified physical-diffusion consistency model.

    Forward signature: UNet(x, y, t, t_prime)
      x       : (B, 1,   H, W)  -- current state at diffusion time t (spin-up or physical)
      y       : (B, C,   H, W)  -- full observation window (NaN = unobserved)
      t       : (B,)            -- current diffusion time
      t_prime : (B,)            -- target diffusion time

    Input channels = 3*C + 1  (x | y_filled | mask_y | w_obs)
    Output         = 1 channel (predicted state at t')

    w_obs is the temporal interpolation weight vector w(t) ∈ [0,1]^C,
    computed internally from t using _compute_obs_weights. It tells the model
    which observation channels are "local" to the current diffusion time:
      - spin-up : w = [1, 0, ..., 0]
      - physical between frames k and k+1: w_k = 1-λ, w_{k+1} = λ, others 0
    This is consistent with the supervised GT target construction in _make_regime_input.
    """
    def __init__(self, config: UNetConfig) -> None:
        super().__init__()
        self.config = config
        C   = config.channels
        top = config.top_blocks_channels[0]

        # input: cat(x:1, y_filled:C, mask_y:C, w_obs:C) = 3C+1 channels
        self.input_projection = nn.Conv2d(3 * C + 1, top, kernel_size=3, padding="same")

        # Separate embeddings for t and t'
        self.time_embedding_t  = TimeEmbedding(config.time_level_channels, config.time_level_scale)
        self.time_embedding_tp = TimeEmbedding(config.time_level_channels, config.time_level_scale)

        self.top_encoder_blocks = self._make_encoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout, self._make_top_block,
        )
        self.mid_encoder_blocks = self._make_encoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout, self._make_mid_block,
        )
        self.mid_decoder_blocks = self._make_decoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout, self._make_mid_block,
        )
        self.top_decoder_blocks = self._make_decoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout, self._make_top_block,
        )
        # Output: 1 channel (single physical state)
        self.output_projection = nn.Conv2d(top, 1, kernel_size=3, padding="same")

    def forward(self, x: Tensor, y: Tensor,
                t: Tensor, t_prime: Tensor) -> Tensor:
        """
        x       : (B, 1,   H, W)
        y       : (B, C,   H, W)   -- NaN where unobserved
        t       : (B,)             -- current diffusion time
        t_prime : (B,)             -- target diffusion time
        """
        B, C, H, W = y.shape
        mask_y = (~torch.isnan(y)).to(dtype=x.dtype)
        y_fill = torch.nan_to_num(y, nan=0.0)

        # Temporal observation weights: (B, C) → broadcast to (B, C, H, W)
        # w(t) encodes which obs channels are local to the current diffusion step.
        # Uses the same interpolation rule as the supervised GT target — consistent by design.
        w_obs = _compute_obs_weights(t.to(dtype=x.dtype), C)          # (B, C)
        w_spatial = w_obs[:, :, None, None].expand(B, C, H, W)        # (B, C, H, W)

        # Concatenate along channel dim and pad to multiple of 8
        inp = torch.cat([x, y_fill, mask_y, w_spatial], dim=1)   # (B, 3C+1, H, W)
        inp, padding = _pad_to_multiple(inp, multiple=8)

        h = self.input_projection(inp)

        emb_t  = self.time_embedding_t(t)
        emb_tp = self.time_embedding_tp(t_prime)
        time_level = torch.cat([emb_t, emb_tp], dim=1)   # (B, 2*TLC, 1, 1)

        top_encoder_embeddings = []
        for block in self.top_encoder_blocks:
            if isinstance(block, UNetBlock):
                h = block(h, time_level)
                top_encoder_embeddings.append(h)
            else:
                h = block(h)

        mid_encoder_embeddings = []
        for block in self.mid_encoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = block(h, time_level)
                mid_encoder_embeddings.append(h)
            else:
                h = block(h)

        for block in self.mid_decoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = torch.cat((h, mid_encoder_embeddings.pop()), dim=1)
                h = block(h, time_level)
            else:
                h = block(h)

        for block in self.top_decoder_blocks:
            if isinstance(block, UNetBlock):
                h = torch.cat((h, top_encoder_embeddings.pop()), dim=1)
                h = block(h, time_level)
            else:
                h = block(h)

        out = self.output_projection(h)
        return _unpad(out, padding)

    # ---- builder helpers ----
    def _make_encoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (ic, oc) in enumerate(zip(channels[:-1], channels[1:])):
            for _ in range(n_blocks[idx]):
                blocks.append(block_fn(ic, oc, dropout[idx]))
                ic = oc
            if has_resampling[idx]:
                blocks.append(Downsample(oc))
        return blocks

    def _make_decoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (oc, ic) in enumerate(list(zip(channels[:-1], channels[1:]))[::-1]):
            if has_resampling[::-1][idx]:
                blocks.append(Upsample(ic))
            inner = []
            for _ in range(n_blocks[::-1][idx]):
                inner.append(block_fn(ic * 2, oc, dropout[::-1][idx]))
                oc = ic
            blocks.extend(inner[::-1])
        return blocks

    def _make_top_block(self, ic, oc, dropout):
        return UNetBlock(ic, oc, 2 * self.config.time_level_channels, dropout)

    def _make_mid_block(self, ic, oc, dropout):
        return UNetBlockWithSelfAttention(
            ic, oc, 2 * self.config.time_level_channels, self.config.n_heads, dropout
        )

    def save_pretrained(self, pretrained_path: str) -> None:
        os.makedirs(pretrained_path, exist_ok=True)
        with open(os.path.join(pretrained_path, "config.json"), mode="w") as f:
            json.dump(asdict(self.config), f)
        torch.save(self.state_dict(), os.path.join(pretrained_path, "model.pt"))

    @classmethod
    def from_pretrained(cls, pretrained_path: str) -> "UNet":
        with open(os.path.join(pretrained_path, "config.json"), mode="r") as f:
            config_dict = json.load(f)
        model = cls(UNetConfig(**config_dict))
        model.load_state_dict(
            torch.load(os.path.join(pretrained_path, "model.pt"), map_location="cpu")
        )
        return model


# ---- Sanity check ----
C = datamodule.window_size
_unet = UNet(UNetConfig(channels=C, cond_channels=2*C)).cpu()
try:
    summary(
        _unet,
        input_size=(
            (1, 1,   100, 100),  # x  (current state, 1 channel)
            (1, C,   100, 100),  # y  (full obs window, C channels)
            (1,),                # t
            (1,),                # t'
        ),
        col_names=["input_size", "output_size", "num_params"],
        verbose=0,
        device="cpu",
    )
except Exception as e:
    n_params = sum(p.numel() for p in _unet.parameters())
    print(f"[torchinfo unavailable: {e}]")
    print(f"UNet parameters: {n_params:,}")

# Quick CPU forward pass — verify shapes and w_obs values
_x = torch.randn(1, 1, 100, 100)
_y = torch.randn(1, C, 100, 100)
_y[0, :, :5, :5] = float("nan")
_t = torch.rand(1)
with torch.no_grad():
    _out = _unet(_x, _y, _t, _t)
print(f"[OK] UNet forward: x {_x.shape} + y {_y.shape} -> {_out.shape}  (3C+1={3*C+1} input channels)")
print(f"[OK] Parameters: {sum(p.numel() for p in _unet.parameters()):,}")

# Verify w_obs for a few representative times
t_IC = 1.0 / C
print(f"\nw_obs sanity check (C={C}, t_IC={t_IC:.3f}):")
for t_test in [0.0, t_IC * 0.5, t_IC, t_IC + 0.01, 0.5, 1.0]:
    w = _compute_obs_weights(torch.tensor([t_test]), C)
    print(f"  t={t_test:.3f}  →  w={w[0].tolist()}  (sum={w[0].sum().item():.3f})")


### LightningModule

Wraps `ConsistencyTrainingDynamicalSystems` (simplified: single GT target, no teacher EMA).

**Two losses**:
1. `L_consistency` — MSE between student output and GT `_make_regime_input(t_next)`, all regimes.
2. `L_obs` — MSE between model output at physical time steps and available observations, weighted by `lambda_obs`.

The teacher EMA is still maintained (for potential future use / EMA student inference), but is no longer called during target construction.

In [ ]:
@dataclass
class LitConsistencyModelConfig:
    initial_ema_decay_rate: float = 0.95
    student_model_ema_decay_rate: float = 0.99993
    lr: float = 1e-4
    betas: Tuple[float, float] = (0.9, 0.995)
    lr_scheduler_start_factor: float = 1e-5
    lr_scheduler_iters: int = 10_000
    total_training_steps: int = 10_000   # fallback only — overridden dynamically in configure_optimizers
    lambda_obs: float = 0.1              # weight for observation fitting loss
    lambda_IC:  float = 1.0              # weight for spinup IC-anchoring loss


class LitConsistencyModel(LightningModule):
    """
    Lightning wrapper for the unified physical-diffusion consistency model.

    Training uses ConsistencyTrainingDynamicalSystems with ANCHOR targets:
      - L_consistency: MSE(student_output, x_{k+1} anchor at t_next) — all regimes
      - L_obs        : MSE(f(x_k, t_k→t_{k+1}), x_{k+1}) on obs mask — λ_obs weighted

    The teacher EMA is maintained (used for spin-up regime) but NOT for physical targets.
    """
    def __init__(
        self,
        consistency_training: ConsistencyTrainingDynamicalSystems,
        student_model: UNet,
        teacher_model: UNet,
        ema_student_model: UNet,
        config: LitConsistencyModelConfig,
    ) -> None:
        super().__init__()
        self.consistency_training = consistency_training
        self.student_model        = student_model
        self.teacher_model        = teacher_model
        self.ema_student_model    = ema_student_model
        self.config               = config
        self.num_timesteps        = consistency_training.initial_timesteps
        # Will be overridden in configure_optimizers with actual training steps
        self._total_training_steps = config.total_training_steps

        for param in self.teacher_model.parameters():
            param.requires_grad = False
        for param in self.ema_student_model.parameters():
            param.requires_grad = False
        self.teacher_model     = self.teacher_model.eval()
        self.ema_student_model = self.ema_student_model.eval()

    # ------------------------------------------------------------------
    # Observation fitting loss
    # ------------------------------------------------------------------
    def _obs_loss(self, batch) -> torch.Tensor:
        """
        For each consecutive physical-frame pair (k → k+1), run the model
        with the clean anchor x_k as input at t_k targeting t_{k+1}, and
        compare to the next GT anchor x_{k+1} on the observed pixels of step k+1.

            L_obs = (1/(C-1)) * Σ_{k=0}^{C-2}
                      ||(f(x_k, t_k, t_{k+1}) - x_{k+1}) ⊙ m_{k+1}||² / (||m_{k+1}||₁ + ε)

        This is CONSISTENT with the anchor-target consistency training:
          • the consistency loss already trains f(interp(t_int), t_int, t_next) → x_{k+1}
          • this loss additionally trains f(x_k_clean, t_k, t_{k+1}) → x_{k+1} on obs pixels
          • starting from a clean anchor (not an interpolated input) adds direct supervision

        Why not use t_0 / self-mapping:
          With anchor training, f(x, t, t') → x_{anchor(t')}.  Using t'=0 asks for the
          anchor at t=0 which is the IC x_0 — useful only at k=0.  Using t'=t makes it a
          trivial identity request, which is not what the model learns.
        """
        x = batch.tgt    # (B, C, H, W)  ground truth
        y = batch.input  # (B, C, H, W)  observations (NaN = missing)
        B, C, H, W = x.shape
        device = x.device

        if C < 2:
            return torch.tensor(0.0, device=device, dtype=x.dtype)

        # NEW design: physical frame k is at diffusion time
        #   t_k = spinup_boundary * (C-1-k) / (C-1)
        # which decreases from spinup_boundary (k=0, IC) to 0 (k=C-1, last frame).
        # The model maps x_k (at t_k) → x_{k+1} (at t_{k+1} < t_k): FORWARD physical time.
        sb = self.consistency_training.spinup_boundary
        total_loss = torch.tensor(0.0, device=device, dtype=x.dtype)

        for k in range(C - 1):
            t_k      = sb * (C - 1 - k)       / max(C - 1, 1)   # decreasing: sb → step
            t_k_next = sb * (C - 2 - k)       / max(C - 1, 1)   # t_{k+1} < t_k

            t_k_full      = torch.full((B,), t_k,      device=device, dtype=x.dtype)
            t_k_next_full = torch.full((B,), t_k_next, device=device, dtype=x.dtype)

            x_k      = x[:, [k],     :, :]   # (B, 1, H, W)  clean input anchor at t_k
            x_k_next = x[:, [k + 1], :, :]   # (B, 1, H, W)  clean target anchor at t_{k+1}

            pred = model_dynamical_systems_forward_wrapper(
                self.student_model,
                x_k, y,
                t_k_full, t_k_next_full,
                self.consistency_training.sigma_min,
                self.consistency_training.sigma_max,
                spinup_boundary=sb,
            )

            # Mask: observed pixels at step k+1 (target frame)
            mask_kp1 = (~torch.isnan(y[:, [k + 1], :, :])).to(dtype=x.dtype)  # (B,1,H,W)
            n_obs    = mask_kp1.sum() + 1e-8

            total_loss = total_loss + (pseudo_huber_loss(pred, x_k_next) * mask_kp1).sum() / n_obs

        return total_loss / (C - 1)

    # ------------------------------------------------------------------
    # Training step
    # ------------------------------------------------------------------

    def _spinup_anchor_loss(self, batch, output) -> torch.Tensor:
        """
        Spinup IC-anchoring loss: L_IC = MSE(g(IC + σ_sp(t)·ε, t_spin, t_IC), IC)

        From noised IC at any spinup time t > spinup_boundary,
        produce clean IC at t_IC = spinup_boundary.
        """
        sb        = self.consistency_training.spinup_boundary
        sigma_min = self.consistency_training.sigma_min
        sigma_max = self.consistency_training.sigma_max

        steps  = output.sigmas
        k_IC   = int((steps > sb).sum().item()) - 1
        if k_IC < 0:
            return batch.tgt.new_zeros(())

        B, C, H, W = batch.tgt.shape
        device     = batch.tgt.device
        dtype      = batch.tgt.dtype
        IC         = batch.tgt[:, [0], :, :]

        t_sp_pool  = steps[:k_IC + 1]
        sp_idx     = torch.randint(0, k_IC + 1, (B,), device=device)
        t_sp       = t_sp_pool[sp_idx]
        t_IC_vec   = torch.full((B,), sb, device=device, dtype=dtype)

        noise      = torch.randn(B, 1, H, W, device=device, dtype=dtype)
        from consistency_models.consistency_models_DynCM import compute_sigma_spinup
        sigma_sp   = compute_sigma_spinup(t_sp, sigma_min, sigma_max, sb)
        noisy_input = IC + sigma_sp.view(B, 1, 1, 1) * noise

        from consistency_models.consistency_models_DynCM import model_dynamical_systems_forward_wrapper
        pred_IC = model_dynamical_systems_forward_wrapper(
            self.student_model, noisy_input, batch.input,
            t_sp, t_IC_vec,
            sigma_min, sigma_max,
            spinup_boundary=sb,
        )
        return pseudo_huber_loss(pred_IC, IC).mean()

    def training_step(self, batch, batch_idx: int):
        if isinstance(batch, list):
            batch = batch[0]

        output = self.consistency_training(
            self.student_model,
            self.teacher_model,
            batch.tgt,
            batch.input,
            self.global_step,
            self._total_training_steps,   # dynamic, updated in configure_optimizers
        )

        self.num_timesteps = output.num_timesteps

        # ── Consistency loss (all regimes, anchor target) ──────────────
        loss_cm = pseudo_huber_loss(
            output.predicted_next_from_intermediate,
            output.target_next_from_current.detach(),
        ).mean()

        # ── Observation fitting loss ───────────────────────────────────
        loss_obs = self._obs_loss(batch)

        # ── IC-anchoring loss (spinup) ──────────────────────────────────
        loss_IC = self._spinup_anchor_loss(batch, output)

        loss = loss_cm + self.config.lambda_obs * loss_obs + self.config.lambda_IC * loss_IC

        # NaN safety: if any component is NaN, return zero loss to skip this batch
        if torch.isnan(loss):
            print(f"[NaN] Ep {self.current_epoch} step {self.global_step}: "
                  f"L_cm={loss_cm.item():.4f} L_IC={loss_IC.item():.4f} L_obs={loss_obs.item():.4f}")
            return torch.tensor(0.0, device=loss.device, dtype=loss.dtype, requires_grad=True)

        is_bad = (loss.item() > 50)
        if batch_idx % 20 == 0 or is_bad:
            d = output.diag or {}
            ds = d.get("student", {})
            dt = d.get("teacher", {})
            def _f(v, fmt=".3f"):
                return f"{v:{fmt}}" if isinstance(v, (int, float)) else str(v)
            tag = "EXPLOSION " if is_bad else ""
            print(
                f"{tag}[Ep {self.current_epoch} | step {self.global_step}]  "
                f"N={self.num_timesteps}  "
                f"L_cm={loss_cm.item():.5f}  "
                f"L_IC={loss_IC.item():.5f}  "
                f"L_obs={loss_obs.item():.5f}  "
                f"loss={loss.item():.5f}  "
                f"spinup/phys={d.get('n_spinup','?')}/{d.get('n_phys','?')}\n"
                f"  student: c_skip=[{_f(ds.get('c_skip_min','?'))},{_f(ds.get('c_skip_max','?'))}]  "
                f"c_out=[{_f(ds.get('c_out_min','?'))},{_f(ds.get('c_out_max','?'))}]  "
                f"model_out_std={_f(ds.get('model_out_std','?'))}  "
                f"result_std={_f(ds.get('result_std','?'))}  "
                f"nan={ds.get('has_nan','?')}\n"
                f"  teacher: c_skip=[{_f(dt.get('c_skip_min','?'))},{_f(dt.get('c_skip_max','?'))}]  "
                f"model_out_std={_f(dt.get('model_out_std','?'))}  "
                f"nan={dt.get('has_nan','?')}\n"
                f"  input_std={_f(d.get('input_std','?'))}  "
                f"target_std={_f(d.get('target_std','?'))}  "
                f"t_curr={d.get('t_curr_range',('?','?'))}  "
                f"t_next={d.get('t_next_range',('?','?'))}"
            )

        self.log_dict({
            "train_loss":      loss,
            "loss_consistency": loss_cm,
            "loss_obs":        loss_obs,
            "num_timesteps":   float(self.num_timesteps),
        }, prog_bar=False)
        return loss

    def on_train_batch_end(self, outputs, batch, batch_idx: int) -> None:
        ema_rate = ema_decay_rate_schedule(
            self.num_timesteps,
            self.config.initial_ema_decay_rate,
            self.consistency_training.initial_timesteps,
        )
        # Check for NaN in student weights before EMA update
        has_nan_weights = any(
            torch.isnan(p).any() for p in self.student_model.parameters() if p.requires_grad
        )
        if has_nan_weights:
            print(f"[FATAL] NaN in student weights at step {self.global_step}!")
            return

        # Log gradient norm
        grad_norm = sum(
            p.grad.norm().item() ** 2
            for p in self.student_model.parameters()
            if p.grad is not None
        ) ** 0.5
        if batch_idx % 20 == 0 or grad_norm > 10:
            print(f"  grad_norm={grad_norm:.4f}")

        update_ema_model_(self.teacher_model,     self.student_model, ema_rate)
        update_ema_model_(self.ema_student_model, self.student_model,
                          self.config.student_model_ema_decay_rate)
        self.log("ema_decay_rate", ema_rate)
        self.log("grad_norm", grad_norm)

    def configure_optimizers(self):
        # ── Dynamic total_training_steps ─────────────────────────────────────────
        # Use the ACTUAL number of optimizer steps that will occur (accounts for
        # accumulate_grad_batches, max_epochs, dataset size).  This ensures N grows
        # from initial_timesteps to final_timesteps over the FULL training run,
        # not just the first ~10k steps.  Without this, N saturates in epoch ~50
        # and the model can learn the trivial identity function (loss→0 but noisy
        # at inference).
        try:
            actual_steps = self.trainer.estimated_stepping_batches
            if actual_steps and int(actual_steps) > 0:
                self._total_training_steps = int(actual_steps)
                print(
                    f"[LitConsistencyModel] Dynamic total_training_steps = "
                    f"{self._total_training_steps}  "
                    f"(config fallback was {self.config.total_training_steps})"
                )
        except Exception as e:
            print(f"[LitConsistencyModel] Could not get estimated_stepping_batches ({e}), "
                  f"using config value {self.config.total_training_steps}")
            self._total_training_steps = self.config.total_training_steps

        opt = torch.optim.Adam(
            self.student_model.parameters(),
            lr=self.config.lr, betas=self.config.betas,
        )
        sched = torch.optim.lr_scheduler.LinearLR(
            opt,
            start_factor=self.config.lr_scheduler_start_factor,
            total_iters=self.config.lr_scheduler_iters,
        )
        return [opt], [{"scheduler": sched, "interval": "step", "frequency": 1}]


## 🚀 Training

### Unified timeline — time grid visualisation

Before training, let's visualise the unified time grid $\{t_0,\dots,t_N\}$ for a given
`num_timesteps` to see how the Karras grid partitions the spin-up and physical regimes.

In [ ]:
from consistency_models.consistency_models_DynCM import compute_sigma_spinup
from consistency_models.utils import karras_schedule

C_vis        = datamodule.window_size
t_IC_vis     = 1.0 / C_vis
# sigma_max=10 calibrated to normalised SPDE data (std≈1, SNR≤0.01).
# physical_steps now = linspace(t_IC, 1.0, C) so green dashes start at t_IC.
SIGMA_MIN    = 0.002
SIGMA_MAX    = 10.0
RHO          = 7.0

for num_t in [5, 9, 17, 50]:
    grid = karras_schedule(num_t, SIGMA_MIN, SIGMA_MAX, RHO, torch.device("cpu"), as_time=True)
    # Physical states are at linspace(t_IC, 1.0, C)  (fixed anchoring)
    physical_ts = torch.linspace(t_IC_vis, 1.0, C_vis)

    n_spinup  = (grid <= t_IC_vis).sum().item()
    n_phys    = (grid > t_IC_vis).sum().item()

    fig, ax = plt.subplots(figsize=(10, 1.4))
    for i, t_val in enumerate(grid.tolist()):
        color = "steelblue" if t_val <= t_IC_vis else "tomato"
        ax.axvline(t_val, color=color, linewidth=2, alpha=0.9)
    for t_phys in physical_ts.tolist():
        ax.axvline(t_phys, color="green", linewidth=1, linestyle="--", alpha=0.7)
    ax.axvline(t_IC_vis, color="black", linewidth=2, linestyle=":", label=f"$t_{{IC}}={t_IC_vis:.2f}$")
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Diffusion time  $t$")
    ax.set_title(
        f"Unified time grid  (num_timesteps={num_t}, σ_max={SIGMA_MAX}) — "
        f"blue=spin-up ({n_spinup}), red=physical ({n_phys}), "
        f"dashed green=physical states $t_k$ (anchored at t_IC)",
        fontsize=9,
    )
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="steelblue", label=f"Spin-up ({n_spinup} pts)"),
        Patch(color="tomato",    label=f"Physical ({n_phys} pts)"),
    ], loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()


### TensorBoard

### Training Loop

In [ ]:
import shutil

# RESET_TRAINING set by the parameters cell above

LOG_DIR     = "logs_CT_DynCM_sic"
CKPT_DIR    = os.path.join(LOG_DIR, "checkpoints")
MODEL_PATH  = os.path.join(LOG_DIR, "best_model")

def _is_valid_checkpoint(path: str) -> bool:
    try:
        ckpt  = torch.load(path, map_location="cpu")
        state = ckpt.get("state_dict", {})
        return all(
            not (torch.is_tensor(v) and torch.isnan(v).any())
            for v in state.values()
        )
    except Exception:
        return False

def find_resume_ckpt(ckpt_dir: str, use_last: bool = True) -> Optional[str]:
    if not os.path.isdir(ckpt_dir):
        return None
    if use_last:
        last = os.path.join(ckpt_dir, "last.ckpt")
        if os.path.isfile(last) and _is_valid_checkpoint(last):
            print(f"[OK] Resuming from: {last}")
            return last
        print("[WARN] last.ckpt missing or invalid — scanning for best valid checkpoint.")
    candidates = sorted(
        [p for p in glob.glob(os.path.join(ckpt_dir, "*.ckpt"))
         if "last" not in os.path.basename(p) and "train_loss=" in os.path.basename(p)],
        key=lambda p: float(os.path.basename(p).split("train_loss=")[-1].replace(".ckpt", "")),
    )
    for p in candidates:
        if _is_valid_checkpoint(p):
            loss_val = float(os.path.basename(p).split("train_loss=")[-1].replace(".ckpt", ""))
            print(f"[OK] Best valid checkpoint: {p}  (train_loss={loss_val:.4f})")
            return p
    print("[WARN] No valid checkpoint found — starting from scratch.")
    return None

if RESET_TRAINING:
    for d in [CKPT_DIR, MODEL_PATH]:
        if os.path.exists(d):
            shutil.rmtree(d)
            print(f"[DELETE] {d}")
    resume_ckpt = None
else:
    resume_ckpt = find_resume_ckpt(CKPT_DIR)

# ---- Models ----
C = datamodule.window_size
student_model     = UNet(UNetConfig(channels=C, cond_channels=2*C))
teacher_model     = UNet(UNetConfig(channels=C, cond_channels=2*C))
ema_student_model = UNet(UNetConfig(channels=C, cond_channels=2*C))
teacher_model.load_state_dict(student_model.state_dict())
ema_student_model.load_state_dict(student_model.state_dict())

# ---- Training ----
# v8: Fixed student/teacher pairwise assignment.
#   • Student takes the BIGGER step (current → next): harder task, gets gradient
#   • Teacher takes the SMALLER step (intermediate → next): reliable target (EMA, no grad)
#   • Matches CM pairwise design. v7 had roles inverted, causing spinup variance
#     explosion as N grew (teacher did the hard step → unreliable targets).
ct = ConsistencyTrainingDynamicalSystems(
    sigma_min=0.002,
    sigma_max=10.0,
    rho=7.0,
    initial_timesteps=5,
    final_timesteps=50,
)

lit_cm = LitConsistencyModel(
    ct,
    student_model,
    teacher_model,
    ema_student_model,
    LitConsistencyModelConfig(
        lr_scheduler_iters=1000,
        total_training_steps=10_000,   # fallback; overridden dynamically in configure_optimizers
        lambda_obs=0.1,
    ),
)

trainer = Trainer(enable_progress_bar=False, 
    accelerator="gpu",
    max_epochs=MAX_EPOCHS,
    accumulate_grad_batches=4,
    precision="bf16-mixed",
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
    log_every_n_steps=1,
    logger=TensorBoardLogger(".", name=LOG_DIR, version=""),
    callbacks=[
        LearningRateMonitor(logging_interval="step"),
        ModelCheckpoint(
            dirpath=CKPT_DIR,
            monitor="train_loss",
            save_top_k=3,
            save_last=True,
            filename="{epoch:03d}-{step}-{train_loss:.4f}",
        ),
    ],
)

import threading, time as _time

_keepalive_stop = threading.Event()
def _keepalive():
    p = os.path.join(os.path.expanduser("~"), ".nfs_keepalive")
    while not _keepalive_stop.wait(60):
        try:
            open(p, "w").close()
        except Exception:
            pass
_keepalive_thread = threading.Thread(target=_keepalive, daemon=True)
_keepalive_thread.start()

seed_everything(42)
trainer.callbacks.append(EpochHeartbeat(every_n_epochs=1))
if SKIP_TRAINING:
    if resume_ckpt is None:
        raise RuntimeError(
            f"SKIP_TRAINING=True but no checkpoint found in {CKPT_DIR} -- "
            "run training first (submit_train.sbatch) before computing metrics."
        )
    print(f'[TRAINING] SKIP_TRAINING=True -- loading weights from {resume_ckpt} directly (trainer.fit() not called)', flush=True)
    _ckpt_state = torch.load(resume_ckpt, map_location='cpu')
    lit_cm.load_state_dict(_ckpt_state['state_dict'])
else:
    print(f'[TRAINING] resume_ckpt={resume_ckpt!r} | MAX_EPOCHS={MAX_EPOCHS}', flush=True)
    trainer.fit(lit_cm, datamodule, ckpt_path=resume_ckpt)

# Save EMA model
lit_cm.ema_student_model.save_pretrained(MODEL_PATH)
print(f"[OK] EMA model saved → {MODEL_PATH}")
_keepalive_stop.set()


## 🎲 Sampling & Evaluation

### Checkpoint Loading

In [ ]:
MODEL_PATH = os.path.join("logs_CT_DynCM_sic", "best_model")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

unet = UNet.from_pretrained(MODEL_PATH).eval().to(device=device, dtype=dtype)
print(f"[OK] Model loaded from: {MODEL_PATH}")
print(f"     Input channels: 3*C+1 = {3 * datamodule.window_size + 1}  (x | y_filled | mask_y | w_obs)")


### Test Batch

In [ ]:
seed_everything(42)
batch = next(iter(datamodule.test_dataloader()))

# m_norm/s_norm needed by the Publication Figures cell below AND the
# Metrics cell (now at the end of the notebook) -- defined once, early,
# here so both can use it regardless of cell order.
m_norm, s_norm = datamodule.norm_stats()

### Ensemble Generation

The sampler traverses the unified time grid from $s=0$ (pure noise) to $s=1$ (clean):

| Phase | Solver time $s$ | Diffusion time $t$ | Behaviour |
|---|---|---|---|
| **Spin-up** | $[0, \text{SPINUP\_FRAC}] \approx [0, 0.3]$ | $[0.7, 1]$ | Noise → IC (frame 0) |
| **Physical** | $[\text{SPINUP\_FRAC}, 1] \approx [0.3, 1]$ | $[0, 0.7]$ | IC → frame 1 → … → frame C−1 |

- `spinup_boundary` $= 1 - \text{SPINUP\_FRAC} = 0.7$ in diffusion time coordinates.
- Physical frame $k$ is anchored at $t_k = \text{sb} \times (C-1-k)/(C-1)$, reached at $s_k = 1 - t_k$.
- With `NSTEPS = 4C+1`, the IC appears at step $\approx C+1$ (solver time $\approx 0.3$) and frames are spaced ~3 steps apart.
- The full observation window $\mathbf{y}$ conditions every step.


In [ ]:
# N_SAMPLES set by the parameters cell above
# NSTEPS = 4*C+1 gives ~C+1 spin-up steps and ~3*C physical steps.
# With SPINUP_BOUNDARY=0.7: spin-up ≈ first 30% of steps (s ∈ [0, 0.3]),
# physical ≈ last 70% (s ∈ [0.3, 1]).  IC appears at s ≈ 0.3 (step ~C+1).
NSTEPS = 4 * datamodule.window_size + 1

SPINUP_BOUNDARY = ct.spinup_boundary   # 0.7 — must match training

STOCHASTIC = False  # False = déterministe (consistency pure, comportement original)
                    # True  = stochastique: réinjecte Δσ·ε à chaque step spin-up
                    #         → débruitage progressif du spin-up (DDPM-like)
consistency_sampling = ConsistencySamplingAndEditingDynamicalSystems(
    sigma_min=0.002,
    sigma_max=10.0,
    spinup_boundary=SPINUP_BOUNDARY,
    stochastic=STOCHASTIC,
)

C  = datamodule.window_size
H, W = batch.tgt.shape[-2], batch.tgt.shape[-1]
y_dev = batch.input.to(device=device, dtype=dtype)

samples         = []
samples_process = []
phys_indices_list = []

with torch.no_grad():
    for i in range(N_SAMPLES):
        noise = torch.randn((1, 1, H, W), device=device, dtype=dtype)
        result = consistency_sampling(
            unet,
            noise,
            y_dev,
            nsteps=NSTEPS,
            clip_denoised=False,
            verbose=(i == 0),
        )
        sample, sample_process, phys_idx = result
        samples.append(sample.float().cpu())
        samples_process.append(sample_process.float().cpu())
        if i == 0:
            phys_indices_list = phys_idx   # list of ints, IC-first order
        print(f"  Sample {i+1}/{N_SAMPLES} done", end="\r")

print(f"\n[OK] {N_SAMPLES} ensemble members generated  (nsteps={NSTEPS})")
print(f"Physical frame indices IC→last: {phys_indices_list}")
print(f"  → solver_times: {[f'{si/(NSTEPS-1):.2f}' for si in phys_indices_list]}")
print(f"  IC expected at s ≈ {1-SPINUP_BOUNDARY:.2f}  (step ~{round((1-SPINUP_BOUNDARY)*(NSTEPS-1))})")
print(f"\nSample shape: {samples[0].shape}")
print(f"Process shape: {samples_process[0].shape}  (nsteps, B=1, 1, H, W)")


### Ensemble


In [ ]:
# Stack samples using the computed physical indices (IC-first order)
def extract_physical_trajectory(proc_tensor: torch.Tensor, idx_list) -> torch.Tensor:
    """proc_tensor: (nsteps+1, 1, 1, H, W) -> (C, H, W)"""
    return proc_tensor[idx_list, 0, 0, :, :]   # (C, H, W)


traj_list  = [extract_physical_trajectory(p, phys_indices_list) for p in samples_process]
traj_stack = torch.stack(traj_list, dim=0).numpy()   # (N, C, H, W)

ens_mean_traj = traj_stack.mean(axis=0)   # (C, H, W)
ens_std_traj  = traj_stack.std(axis=0)    # (C, H, W)
gt_np         = batch.tgt[0].float().numpy()   # (C, H, W)

### Publication Figures — comparison & uncertainty


In [ ]:
from matplotlib.gridspec import GridSpec

FIG_TAG = 'DynCM_sic'
FIG_DIR = os.path.join('figures', FIG_TAG)
os.makedirs(FIG_DIR, exist_ok=True)

C = datamodule.window_size
ws = C

obs_phys  = batch.input[0].float().numpy()   * s_norm + m_norm   # (C,H,W)
gt_phys   = batch.tgt[0].float().numpy()     * s_norm + m_norm   # (C,H,W)
mean_phys = ens_mean_traj * s_norm + m_norm                         # (C,H,W)
std_phys  = ens_std_traj  * s_norm                                  # (C,H,W)
mbr0_phys = traj_stack[0]  * s_norm + m_norm                        # (C,H,W)
mbr1_phys = traj_stack[1]  * s_norm + m_norm                        # (C,H,W)

# Valid-pixel mask: keep only pixels where ASIP has >= 1 valid obs over
# the window (the dataset's land_mask and lat/lon are both unreliable,
# so derive validity directly from ASIP coverage instead of geolocation).
is_land = ~np.isfinite(gt_phys).any(axis=0)
print(f'Masked-out fraction (no ASIP obs in window): {is_land.mean()*100:.1f}%')
for _arr in (obs_phys, gt_phys, mean_phys, std_phys, mbr0_phys, mbr1_phys):
    _arr[:, is_land] = np.nan

# SIC is a physical fraction in [0,1] -- Blues_r (not RdBu_r/symmetric, which
# was copied from the GP/SPDE reference notebook where the field is a
# zero-centered anomaly, not a bounded [0,1] concentration).
SIC_MAX = 1.0
vmax_s = float(np.nanpercentile(std_phys, 99))

cmap_f = plt.cm.Blues_r.copy(); cmap_f.set_bad('lightgray')
cmap_s = plt.cm.Reds.copy();   cmap_s.set_bad('lightgray')

def _save_strip(data, filename, vmin, vmax, cmap):
    FW, FH, CB_H = 2.0, 2.0, 0.28
    fig_w = C * FW
    fig_h = FH + CB_H + 0.06
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = GridSpec(2, C,
                  left=0.01, right=0.99, top=0.99, bottom=0.01,
                  height_ratios=[FH, CB_H], hspace=0.06, wspace=0.03)
    for c in range(C):
        ax = fig.add_subplot(gs[0, c])
        ax.imshow(data[c], origin='lower', cmap=cmap,
                  vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.axis('off')
    ax_cb = fig.add_subplot(gs[1, :])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax_cb, orientation='horizontal')
    cb.ax.tick_params(labelsize=9)
    fpath = os.path.join(FIG_DIR, f'{filename}.png')
    fig.savefig(fpath, dpi=200, bbox_inches='tight')
    print(f'  Saved: {fpath}')
    plt.show()

figures = [
    (obs_phys,  f'{FIG_TAG}_obs',        0, SIC_MAX, cmap_f),
    (gt_phys,   f'{FIG_TAG}_gt',         0, SIC_MAX, cmap_f),
    (mean_phys, f'{FIG_TAG}_dyncm_mean', 0, SIC_MAX, cmap_f),
    (std_phys,  f'{FIG_TAG}_dyncm_spread', 0.0, vmax_s, cmap_s),
    (mbr0_phys, f'{FIG_TAG}_member0',    0, SIC_MAX, cmap_f),
    (mbr1_phys, f'{FIG_TAG}_member1',    0, SIC_MAX, cmap_f),
]

for data, fname, vmin, vmax, cmap in figures:
    _save_strip(data, fname, vmin, vmax, cmap)

### Transport process — sampling trajectory visualisation

Display **all diffusion steps** and then extract the **physical time steps** from the trajectory.  
The model maps noise at $t=1$ through the spin-up into the physical regime step by step.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

# ── Regime fractions ───────────────────────────────────────────────────────────
SPINUP_FRAC = round(1.0 - consistency_sampling.spinup_boundary, 2)   # 0.30 (not 0.300000004)

proc  = samples_process[0]
nfr   = proc.shape[0]

def solver_time(si):
    return round(si / max(nfr - 1, 1), 2)

phys_frames    = proc[phys_indices_list, 0, 0, :, :].numpy()
phys_times_lst = [solver_time(i) for i in phys_indices_list]
gt_phys_frames = batch.tgt[0].float().numpy()
obs_frames     = np.nan_to_num(batch.input[0].float().numpy(), nan=0.0)

vmin_all, vmax_all = -2, 2

# ── Build step_indices ─────────────────────────────────────────────────────────
# 4 evenly-spaced spin-up steps (0 … phys0-1) then, for each pair of consecutive
# physical frames, interleave: [phys_k, mid_k_k+1, phys_{k+1}, ...]
phys0        = phys_indices_list[0]
N_SPINUP_COL = 4
spinup_steps = sorted({
    round(k * (phys0 - 1) / max(N_SPINUP_COL - 1, 1))
    for k in range(N_SPINUP_COL)
    if round(k * (phys0 - 1) / max(N_SPINUP_COL - 1, 1)) < phys0
})

phys_and_intermed = []
for k, pi in enumerate(phys_indices_list):
    phys_and_intermed.append(pi)
    if k < len(phys_indices_list) - 1:
        mid = int((pi + phys_indices_list[k + 1]) / 2)   # int() avoids banker's rounding
        phys_and_intermed.append(mid)

step_indices         = spinup_steps + phys_and_intermed
step_indices_display = step_indices
n_top                = len(step_indices_display)

phys_step_set = set(phys_indices_list)
spinup_cols   = [j for j, si in enumerate(step_indices_display)
                 if si not in phys_step_set and solver_time(si) <= SPINUP_FRAC]

# Physical columns — guaranteed to be exact (values are in step_indices)
phys_col = [step_indices_display.index(pi) for pi in phys_indices_list]

print(f"step_indices ({n_top} cols) = {step_indices}")
print(f"phys_col                   = {phys_col}")
print(f"spinup_cols                = {spinup_cols}")

# ── Figure layout ──────────────────────────────────────────────────────────────
# 4 rows (transport | prediction | GT | obs) + horizontal colorbar at bottom
FIG_W   = max(n_top * 1.55, 18)
FIG_H   = 11.5
PUB_FS  = 11   # publication font size for row labels
TOP_FS  = 10   # font size for diffusion-time titles in top strip

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs  = fig.add_gridspec(
    4, n_top,
    height_ratios=[3, 2, 2, 2],
    hspace=0.25, wspace=0.04,
    left=0.10, right=0.98, top=0.93, bottom=0.14,
)

axes_top = [fig.add_subplot(gs[0, j]) for j in range(n_top)]
axes_mid = {k: fig.add_subplot(gs[1, col]) for k, col in enumerate(phys_col)}
axes_bot = {k: fig.add_subplot(gs[2, col]) for k, col in enumerate(phys_col)}
axes_obs = {k: fig.add_subplot(gs[3, col]) for k, col in enumerate(phys_col)}

# ── Row 0: transport strip ────────────────────────────────────────────────────
for j, si in enumerate(step_indices_display):
    ax    = axes_top[j]
    s_val = solver_time(si)
    ax.imshow(proc[si, 0, 0, :, :].numpy(), origin="lower",
              cmap="RdBu_r", vmin=vmin_all, vmax=vmax_all, interpolation="nearest")
    col_t = "steelblue" if s_val <= SPINUP_FRAC else "tomato"
    ax.set_title(f"s={s_val:.2f}", fontsize=TOP_FS, color=col_t, pad=2, fontweight="bold")
    ax.axis("off")
    if si in phys_step_set:
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor("limegreen"); sp.set_linewidth(2.5)

# ── Rows 1–3: prediction / GT / obs (blank at intermediate columns) ───────────
C = len(phys_indices_list)
for k in range(C):
    axes_mid[k].imshow(phys_frames[k], origin="lower", cmap="RdBu_r",
                       vmin=vmin_all, vmax=vmax_all, interpolation="nearest")
    axes_mid[k].axis("off")
    for sp in axes_mid[k].spines.values():
        sp.set_visible(True); sp.set_edgecolor("limegreen"); sp.set_linewidth(2)

    axes_bot[k].imshow(gt_phys_frames[k], origin="lower", cmap="RdBu_r",
                       vmin=vmin_all, vmax=vmax_all, interpolation="nearest")
    axes_bot[k].axis("off")

    axes_obs[k].imshow(obs_frames[k], origin="lower", cmap="RdBu_r",
                       vmin=vmin_all, vmax=vmax_all, interpolation="nearest")
    axes_obs[k].axis("off")

# ── Row labels (publication-ready: black, bold, large) ───────────────────────
fig.canvas.draw()
label_x = axes_top[0].get_position().x0 - 0.012
rows_meta = [
    (axes_top[0],  "Transport\nprocess"),
    (axes_mid[0],  "DynCM\nprediction"),
    (axes_bot[0],  "Ground\ntruth"),
    (axes_obs[0],  "Observations"),
]
for ax_ref, lbl in rows_meta:
    pos = ax_ref.get_position()
    yc  = (pos.y0 + pos.y1) / 2
    fig.text(label_x, yc, lbl, va="center", ha="right",
             fontsize=PUB_FS, color="black", fontweight="bold", rotation=0,
             fontfamily="DejaVu Sans")

# ── Arrows: transport strip → prediction row ──────────────────────────────────
for k, col in enumerate(phys_col):
    sb = axes_top[col].get_position()
    db = axes_mid[k].get_position()
    xm = (sb.x0 + sb.x1) / 2
    fig.add_artist(mpatches.FancyArrowPatch(
        (xm, sb.y0 - 0.003), (xm, db.y1 + 0.003),
        transform=fig.transFigure, arrowstyle="-|>",
        color="blueviolet", mutation_scale=10, linewidth=1.5,
    ))

# ── Spin-up brace above top strip ────────────────────────────────────────────
if spinup_cols:
    p0   = axes_top[spinup_cols[0]].get_position()
    p1   = axes_top[spinup_cols[-1]].get_position()
    xL, xR = p0.x0, p1.x1
    yT   = p0.y1 + 0.016
    th   = 0.007
    for xt in [xL, xR]:
        fig.add_artist(plt.Line2D([xt, xt], [yT, yT - th],
            transform=fig.transFigure, color="steelblue", linewidth=1.4, clip_on=False))
    fig.add_artist(plt.Line2D([xL, xR], [yT, yT],
        transform=fig.transFigure, color="steelblue", linewidth=1.4, clip_on=False))
    fig.text((xL + xR) / 2, yT + 0.003,
             f"Spin-up  (s ≤ {SPINUP_FRAC:.2f})",
             ha="center", va="bottom", fontsize=PUB_FS - 1, color="steelblue")

# ── Horizontal colorbar at bottom ────────────────────────────────────────────
cbar_ax = fig.add_axes([0.15, 0.04, 0.70, 0.018])
sm = plt.cm.ScalarMappable(cmap="RdBu_r",
                            norm=plt.Normalize(vmin=vmin_all, vmax=vmax_all))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Gaussian Process", fontsize=PUB_FS + 1, fontweight="bold")
cbar.ax.tick_params(labelsize=PUB_FS - 1)

fpath = os.path.join(FIG_DIR, f'DynCM_sic_transport.png')
fig.savefig(fpath, dpi=200, bbox_inches='tight')
print(f'  Saved: {fpath}')
plt.show()

# ── Variance collapse ─────────────────────────────────────────────────────────
step_stds = np.array([proc[k, 0, 0, :, :].numpy().std() for k in range(nfr)])
gt_std    = batch.tgt[0].float().numpy().std()
fig2, ax2 = plt.subplots(figsize=(8, 3))
s_vals = np.array([solver_time(k) for k in range(nfr)])
ax2.plot(s_vals, step_stds, marker="o", color="dimgray", label="Member 0")
ax2.axhline(gt_std, color="green", linestyle="--", label=f"GT std = {gt_std:.3f}")
ax2.axvspan(0.0, SPINUP_FRAC, alpha=0.10, color="steelblue",
            label=f"Spin-up (s ≤ {SPINUP_FRAC:.2f})")
ax2.axvspan(SPINUP_FRAC, 1.0, alpha=0.10, color="tomato", label="Physical regime")
for pi, pt in zip(phys_indices_list, phys_times_lst):
    ax2.axvline(pt, color="limegreen", alpha=0.6, linewidth=1)
ax2.set_xlabel("Solver time  s  (0 = pure noise, 1 = clean)", fontsize=PUB_FS)
ax2.set_ylabel("Spatial std", fontsize=PUB_FS)
ax2.set_title("Variance collapse — spin-up → physical", fontsize=PUB_FS + 1)
ax2.legend(fontsize=PUB_FS - 1)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
fpath2 = os.path.join(FIG_DIR, f'DynCM_sic_variance_collapse.png')
fig2.savefig(fpath2, dpi=200, bbox_inches='tight')
print(f'  Saved: {fpath2}')
plt.show()
print(f"Std step 0 (noise): {step_stds[0]:.4f}   Std last step: {step_stds[-1]:.4f}   GT std: {gt_std:.4f}")

## 📊 Metrics

- **μ-score** = $1 - \text{RMSE}/\sigma_\text{GT}$ (higher is better), computed on the
  Marginal Ice Zone (MIZ, 15-85% concentration) only
- **RMSE** (lower is better), MIZ only
- **Spread-skill ratio** (ideal ~ 1)
- **Radial PSD** — spectral comparison vs GT (full field, PSD can't be
  restricted to a scattered pixel mask)

In [ ]:
import sys
sys.path.append('../..')   # -> consistency/
from spectral_utils import radial_psd_2d, psd_spectral_score, resolved_scale
import pandas as pd

try:
    from properscoring import crps_ensemble as crps_ens_fn
    HAS_PROPERSCORING = True
except ImportError:
    HAS_PROPERSCORING = False
    print("⚠️  properscoring not installed — CRPS disabled.")

DX_PX  = 1.0                           # 1 pixel = 1 km (assumption)

# Marginal Ice Zone (MIZ) bounds -- standard glaciology convention (15-85%
# concentration). Pointwise metrics (Score/RMSE/sigma/CRPS) computed over the
# WHOLE domain are dominated by the huge near-constant open-water/full-ice
# background -- restricting to the MIZ targets the only part of the field
# with real reconstruction difficulty. lambda_x (spectral resolved scale)
# still needs the FULL 2D field (PSD/FFT can't be restricted to a scattered
# pixel mask), so it is computed per (sample, timestep) on the whole domain
# and then aggregated like the other metrics, not restricted itself.
MIZ_LO, MIZ_HI = 0.15, 0.85

m_norm, s_norm = datamodule.norm_stats()

# ── Full test-set evaluation ─────────────────────────────────────────────
# Loops over EVERY batch of datamodule.test_dataloader() (not just the first
# one used for the illustrative figures above) and EVERY timestep of the
# assimilation window (not just T_EVAL=window_size//2), regenerating a fresh
# ensemble each time via the DynCM boundary-aware consistency sampler. This is
# the expensive part (N_SAMPLES x n_test_samples sampling calls) -- N_SAMPLES
# was reduced from 50 to 20 in methods.yaml specifically to keep a
# full-test-set pass affordable.
_all = {'score': [], 'rmse': [], 'sigma_gt': [], 'sigma_pred': [], 'crps': [], 'lambda_x': []}
_n_pairs = 0

seed_everything(42)
for _tb in datamodule.test_dataloader():
    _gt_b = _tb.tgt.to(device=device, dtype=dtype)      # (B, C, H, W) NaN ok
    _y_b  = _tb.input.to(device=device, dtype=dtype)     # (B, C, H, W)
    _Bb, _Cb, _Hb, _Wb = _gt_b.shape
    _valid_b   = ~torch.isnan(_gt_b)
    _obsmask_b = ~torch.isnan(_y_b)

    for _bi in range(_Bb):
        _samples_process = []
        _phys_idx = None
        with torch.no_grad():
            for _s in range(N_SAMPLES):
                _noise = torch.randn((1, 1, _Hb, _Wb), device=device, dtype=dtype)
                _sample, _sample_process, _phys_idx = consistency_sampling(
                    unet, _noise, _y_b[_bi:_bi + 1],
                    nsteps=NSTEPS, clip_denoised=False, verbose=False,
                )
                _samples_process.append(_sample_process.float().cpu())

        _traj_list  = [extract_physical_trajectory(p, _phys_idx) for p in _samples_process]
        _ens_b      = torch.stack(_traj_list, dim=0).numpy()   # (N, C, H, W) normalised
        _ens_mean_b = _ens_b.mean(axis=0)                       # (C, H, W)

        for _t in range(_Cb):
            _valid_t = _valid_b[_bi, _t].cpu().numpy()
            if _valid_t.sum() < 10:
                continue
            _gt_p   = _gt_b[_bi, _t].float().cpu().numpy() * s_norm + m_norm
            _mean_p = _ens_mean_b[_t]                      * s_norm + m_norm
            _ens_p  = _ens_b[:, _t]                        * s_norm + m_norm
            _miz_t  = _valid_t & (_gt_p > MIZ_LO) & (_gt_p < MIZ_HI)
            if _miz_t.sum() < 5:
                continue

            _gt_v, _pred_v = _gt_p[_miz_t], _mean_p[_miz_t]
            _sigma = float(np.std(_gt_v))
            if _sigma <= 0:
                continue
            _rmse  = float(np.sqrt(np.mean((_pred_v - _gt_v) ** 2)))
            _score = 1.0 - _rmse / _sigma
            _sigma_pred = float(np.std(_pred_v))

            _gt_f   = np.nan_to_num(_gt_p,   nan=0.0)
            _pred_f = np.nan_to_num(_mean_p, nan=0.0)
            _wl, _, _, _spec = psd_spectral_score(_pred_f, _gt_f, dx=DX_PX)
            _lam = resolved_scale(_wl, _spec, threshold=0.5)

            _crps_val = np.nan
            if HAS_PROPERSCORING:
                _obs_t  = _obsmask_b[_bi, _t].cpu().numpy()
                _obs_ij = np.argwhere(_miz_t & _obs_t)
                if len(_obs_ij) > 0:
                    _stride = max(1, len(_obs_ij) // 50)   # subsample -- CRPS is O(n) Python loop
                    _crps_val = float(np.mean([
                        crps_ens_fn(float(_gt_p[i, j]), _ens_p[:, i, j])
                        for i, j in _obs_ij[::_stride]
                    ]))

            _all['score'].append(_score)
            _all['rmse'].append(_rmse)
            _all['sigma_gt'].append(_sigma)
            _all['sigma_pred'].append(_sigma_pred)
            _all['crps'].append(_crps_val)
            _all['lambda_x'].append(_lam)
            _n_pairs += 1

print(f"Evaluated {_n_pairs} (test sample, timestep) pairs with MIZ coverage across the full test set")

def _agg(vals):
    a = np.asarray(vals, dtype=float)
    a = a[~np.isnan(a)]
    return (float(np.mean(a)), float(np.std(a))) if len(a) else (np.nan, np.nan)

_score_m, _score_s = _agg(_all['score'])
_rmse_m, _rmse_s = _agg(_all['rmse'])
_sgt_m, _sgt_s = _agg(_all['sigma_gt'])
_spr_m, _spr_s = _agg(_all['sigma_pred'])
_crps_m, _crps_s = _agg(_all['crps'])
_lam_m, _lam_s = _agg(_all['lambda_x'])

row_dyncm = {
    'Method'   : 'DynCM (ens. mean, MIZ, full test set)',
    'Score ↑'  : f'{_score_m:.3f} ± {_score_s:.3f}',
    'RMSE ↓'   : f'{_rmse_m:.4f} ± {_rmse_s:.4f}',
    'σ_GT'     : f'{_sgt_m:.4f} ± {_sgt_s:.4f}',
    'σ_pred'   : f'{_spr_m:.4f} ± {_spr_s:.4f}',
    'CRPS ↓'   : f'{_crps_m:.4f} ± {_crps_s:.4f}' if not np.isnan(_crps_m) else '--',
    'λx [px]'  : f'{_lam_m:.1f} ± {_lam_s:.1f}' if not np.isnan(_lam_m) else '?',
}
df_metrics = pd.DataFrame([row_dyncm]).set_index('Method')
df_metrics.columns = ['Score ↑','RMSE ↓','σ_GT','σ_pred','CRPS ↓','λx [px]']
print(f'\n## Metrics -- full test set, MIZ only ({MIZ_LO}-{MIZ_HI}), n={_n_pairs} (sample,t) pairs\n')

# ── Canonicalize columns for the cross-method LaTeX table (make_latex_table.py) ──
# Every notebook in the suite must expose the SAME column names (RMSE,
# lambda_x, CRPS) regardless of internal naming (unicode arrows/sigma vs
# plain ascii, [px]/[deg] unit suffixes) -- otherwise make_latex_table.py's
# column-union logic creates duplicate columns (e.g. both "RMSE" and
# "RMSE ↓") instead of one shared column per metric. Score/sigma_GT/sigma_pred
# are dropped (not part of the target table). "±" is replaced with the
# LaTeX-safe "$\pm$" so the aggregated .tex table compiles cleanly.
_col_map = {
    'RMSE': 'RMSE', 'RMSE ↓': 'RMSE', 'RMSE down': 'RMSE',
    'lambda_x': 'lambda_x', 'lambda_x [px]': 'lambda_x', 'lambda_x [deg]': 'lambda_x',
    'lambda_x px': 'lambda_x', 'lambda_x [km]': 'lambda_x',
    'λx [px]': 'lambda_x', 'λx [deg]': 'lambda_x', 'λx px': 'lambda_x', 'λx [km]': 'lambda_x',
    'CRPS': 'CRPS', 'CRPS ↓': 'CRPS', 'CRPS down': 'CRPS',
}
df_metrics = df_metrics.rename(columns=_col_map)
for _c in df_metrics.columns:
    df_metrics[_c] = df_metrics[_c].apply(lambda v: v.replace('±', '$\\pm$') if isinstance(v, str) else v)
_keep = [c for c in ['RMSE', 'lambda_x', 'CRPS'] if c in df_metrics.columns]
df_metrics = df_metrics[_keep]

display(df_metrics)

# ── Illustrative PSD plot (single example, T_EVAL=window_size//2 of the first
# test batch, same one used by the figures above) -- qualitative check only,
# NOT the quantitative table (that's df_metrics above, full test set now).
T_EVAL = datamodule.window_size // 2
gt_n_ex   = batch.tgt[0, T_EVAL].float().numpy()
gt_p_ex   = gt_n_ex * s_norm + m_norm
mean_n_ex = ens_mean_traj[T_EVAL]
mean_p_ex = mean_n_ex * s_norm + m_norm

wl_gt,    psd_gt_sig    = radial_psd_2d(gt_p_ex,   dx=DX_PX)
wl_dyncm2, psd_dyncm_sig = radial_psd_2d(mean_p_ex, dx=DX_PX)
wl_dyncm, _, _, spec_dyncm = psd_spectral_score(mean_p_ex, gt_p_ex, dx=DX_PX)
lambda_x_dyncm = resolved_scale(wl_dyncm, spec_dyncm)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for wl, psd, color, ls, lbl in [
    (wl_gt,     psd_gt_sig,    'black', '-',  'GT'),
    (wl_dyncm2, psd_dyncm_sig, 'C0',   '-',  'DynCM (ens. mean)'),
]:
    v = np.isfinite(wl) & np.isfinite(psd) & (wl > 0) & (psd > 0)
    ax.loglog(1.0/wl[v], psd[v], color=color, ls=ls, lw=2, label=lbl)
ax.set_xlabel('Spatial frequency  [cycles / pixel]')
ax.set_ylabel('PSD')
ax.set_title(f'Radial PSD (Hann, illustrative t={T_EVAL})')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

ax = axes[1]
lbl_dyncm = f'DynCM  (λx = {lambda_x_dyncm:.1f} px)' if not np.isnan(lambda_x_dyncm) else 'DynCM'
v = np.isfinite(wl_dyncm) & np.isfinite(spec_dyncm)
ax.plot(wl_dyncm[v], spec_dyncm[v], color='C0', ls='-', lw=2, label=lbl_dyncm)
ax.axhline(0.5, color='gray', lw=1.2, ls='--', label='threshold 0.5')
if not np.isnan(lambda_x_dyncm):
    ax.axvline(lambda_x_dyncm, color='C0', lw=1, ls=':')
ax.set_xlabel('Wavelength [px]')
ax.set_ylabel('Spectral score')
ax.set_title('Score PSD  = 1 − PSD(err) / PSD(GT) (illustrative)')
ax.set_ylim(-0.3, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- metrics serialization (patch_notebook_params.py) ---
import os
os.makedirs(os.path.dirname(METRICS_CSV) or '.', exist_ok=True)
_df_out = df_metrics.reset_index() if df_metrics.index.name == 'Method' else df_metrics
_df_out.to_csv(METRICS_CSV, index=False)
print(f'Metrics written to {METRICS_CSV}')
